In [3]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("VisualizationEngine")

class GenesisVisualizerSuite:
    """
    Enterprise-grade interactive visualization engine utilizing Plotly
    to map macro-market and micro-operational metrics from the startup ledger.
    """
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.template = "plotly_dark"
        os.makedirs("reports", exist_ok=True)

    def generate_all_visualizations(self):
        logger.info("Initializing multi-dimensional visualization suite...")

        # 1. VALUATION DENSITY DISTRIBUTION
        logger.info("Generating Chart 1: Valuation Density...")
        fig1 = px.histogram(
            self.df, x='Valuation', color='Success_Label', nbins=100, log_y=True,
            title='Enterprise Structural Valuation Densities (Log Scaled)',
            template=self.template, color_discrete_sequence=px.colors.qualitative.G10
        )
        fig1.update_layout(xaxis_title="Valuation ($)", yaxis_title="Count (Log Scale)", bargap=0.1)
        fig1.write_json("reports/chart_valuation_dist.json")

        # 2. GLOBAL CAPITAL DISPERSAL MATRIX MAP
        logger.info("Generating Chart 2: Global Capital Allocation...")
        geo_df = self.df.groupby('Country')['Total Funding'].agg(['sum', 'count']).reset_index()
        fig2 = px.choropleth(
            geo_df, locations='Country', locationmode='country names', color='sum',
            hover_data=['count'], title='Global Capital Dispersal Matrix Map',
            template=self.template, color_continuous_scale=px.colors.sequential.Plasma
        )
        fig2.update_layout(coloraxis_colorbar=dict(title="Total Funding ($)"))
        fig2.write_json("reports/chart_global_funding.json")

        # 3. VERTICAL ECOSYSTEM TREEMAP
        logger.info("Generating Chart 3: Industry Capital Distribution...")
        fig3 = px.treemap(
            self.df, path=[px.Constant("All Ecosystems"), 'Industry', 'Sub Industry'],
            values='Total Funding', title='Capital Distribution Across Operational Domains',
            template=self.template, color='Industry', color_discrete_sequence=px.colors.qualitative.Pastel
        )
        fig3.write_json("reports/chart_industry_tree.json")

        # 4. FINANCIAL PEARSON CORRELATION MATRIX MAP
        logger.info("Generating Chart 4: Feature Correlation Heatmap...")
        corr_cols = ['Total Funding', 'Monthly Revenue', 'Monthly Burn Rate', 'ARR', 'Valuation', 'Team Size', 'LTV', 'CAC']
        corr_matrix = self.df[corr_cols].corr()
        fig4 = px.imshow(
            corr_matrix, text_auto=".2f", title='Financial Feature Pearson Matrix Map',
            template=self.template, color_continuous_scale=px.colors.sequential.Cividis
        )
        fig4.write_json("reports/chart_correlation.json")

        # 5. BURN OPTIMIZATION VS PERFORMANCE SCALABILITY TOPOLOGY
        logger.info("Generating Chart 5: Burn Rate vs Revenue Growth Bubble Matrix...")
        # Sample down slightly for plot responsiveness
        sample_df = self.df.sample(n=min(2000, len(self.df)), random_state=42)
        fig5 = px.scatter(
            sample_df, x='Monthly Burn Rate', y='Revenue Growth', size='Valuation',
            color='Success_Label', hover_name='Company_Name',
            title='Burn Optimization vs Performance Scalability Topologies (Sampled)',
            template=self.template, color_discrete_sequence=px.colors.qualitative.Bold
        )
        fig5.update_layout(xaxis_title="Monthly Burn Rate ($)", yaxis_title="Revenue Growth Rate (%)")
        fig5.write_json("reports/chart_bubble_burn.json")

        # 6. SAAS UNIT ECONOMICS RADAR ALIGNMENT (AVERAGES)
        logger.info("Generating Chart 6: Industry Multi-Axis Radar...")
        industry_metrics = self.df.groupby('Industry')[['Founder Technical Score', 'Founder Leadership Score', 'AI Usage Score', 'SEO Score', 'Innovation Score']].mean().reset_index()

        fig6 = go.Figure()
        categories = ['Founder Tech', 'Founder Leadership', 'AI Usage', 'SEO Score', 'Innovation']

        for idx, row in industry_metrics.iterrows():
            fig6.add_trace(go.Scatterpolar(
                r=[row['Founder Technical Score'], row['Founder Leadership Score'], row['AI Usage Score'], row['SEO Score'], row['Innovation Score']],
                theta=categories,
                fill='toself',
                name=row['Industry']
            ))

        fig6.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
            showlegend=True, title="Strategic Score Vectors by Industry Domain", template=self.template
        )
        fig6.write_json("reports/chart_industry_radar.json")

        logger.info("All interactive data intelligence visualization artifacts successfully rendered and saved to /reports.")

        # Render the master composite dashboard view inside Colab cell output
        fig1.show()
        fig5.show()

# Execution instantiation
visual_engine = GenesisVisualizerSuite(startup_df)
visual_engine.generate_all_visualizations()